# Détection des Zones d'Altération Hydrothermale

Ce notebook utilise les indices spectraux SWIR et le modèle Prithvi pour cartographier les zones d'altération (argiles, oxydes de fer).

In [ ]:
!pip install geemap earthengine-api rasterio terratorch torch matplotlib -q
import ee, geemap, torch, rasterio, os
import numpy as np
import matplotlib.pyplot as plt

ee.Initialize(project='geocongoai-api')

In [ ]:
roi = ee.Geometry.Rectangle([15.2, -4.8, 15.6, -4.4])
collection = (ee.ImageCollection("COPERNICUS/S2_SR_HARMONIZED")
              .filterBounds(roi)
              .filterDate('2023-01-01', '2023-12-31')
              .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', 10)))

image = collection.median().clip(roi)
geemap.ee_export_image(image.select(['B2', 'B3', 'B4', 'B8', 'B11', 'B12']), 'input.tif', scale=30, region=roi)

In [ ]:
with rasterio.open('input.tif') as src:
    bands = src.read().astype(np.float32)
    
red = bands[2]
swir1 = bands[4]
swir2 = bands[5]

clay_idx = swir1 / (swir2 + 1e-8)
iron_idx = swir1 / (red + 1e-8)
alteration = (clay_idx + iron_idx) / 2.0

plt.figure(figsize=(10, 6))
plt.imshow(alteration, cmap='hot')
plt.colorbar(label='Indice d\'altération')
plt.title("Zones d'Altération Hydrothermale")
plt.show()